# Mathematical Language for Quantum Computing
Chapter 3 companion. Predict each answer, calculate by hand, run the cell and explain the result. This notebook uses NumPy for mathematical calculations and Matplotlib for figures. No hardware account or Qiskit installation is needed.


In [ ]:
%pip -q install "numpy>=2,<3" "matplotlib>=3.8,<4"


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("NumPy", np.__version__)


## 3.1 Complex arithmetic
Do not confuse a square with a squared magnitude. NumPy uses `1j` for the imaginary unit. Predict both values for (1+2i)/3.


In [ ]:
z = (1 + 2j) / 3
print("Conjugate:", z.conjugate())
print("Square:", z**2, "Squared magnitude:", abs(z)**2)
assert np.isclose(abs(z)**2, 5/9)
assert np.isclose(z**2, (-3+4j)/9)
fig, ax = plt.subplots()
for w, label in [(z, "z"), (z.conjugate(), "conjugate")]:
    ax.plot([0,w.real], [0,w.imag], marker="o", label=label)
ax.axhline(0, color="grey", linewidth=0.5)
ax.axvline(0, color="grey", linewidth=0.5)
ax.set(xlabel="Real", ylabel="Imaginary", title="One complex number and its conjugate")
ax.set_aspect("equal")
ax.legend()
plt.show()


## 3.2 Dot products and normalization
With u=(1,1), project v=(1,2) onto u. The denominator matters because u is not unit length.


In [ ]:
u = np.array([1., 1.])
v = np.array([1., 2.])
projection = (u @ v) / (u @ u) * u
residual = v - projection
print("Projection:", projection, "Residual:", residual)
assert np.allclose(projection, [1.5, 1.5])
assert np.isclose(u @ residual, 0)
def normalize(v):
    v = np.asarray(v, dtype=complex)
    length = np.linalg.norm(v)
    if length == 0:
        raise ValueError("Cannot normalize zero")
    return v / length
print(normalize([3,4]))
try:
    normalize([0,0])
except ValueError as e:
    print(e)
else:
    raise AssertionError("Zero normalization should fail")


## 3.3 Inner products and bras
`np.vdot(u,v)` conjugates u. For one-dimensional arrays, `.T` does not change shape; use `.conj()` for the conjugation and reshape when a row or column is required.


In [ ]:
u = np.array([1, 1j]) / np.sqrt(2)
v = np.array([1j, 1]) / np.sqrt(2)
print("Incorrect non-conjugated product:", np.dot(u,v))
print("Inner product:", np.vdot(u,v))
assert np.isclose(np.vdot(u,v), 0)
assert np.isclose(np.vdot(u,u), 1)
bra = u.conj().reshape(1,-1)
ket = u.reshape(-1,1)
print("Bra shape:", bra.shape, "Ket shape:", ket.shape)
assert np.allclose(bra @ ket, [[1]])


## 3.4 Outer products and projectors
`np.outer` does not conjugate its second argument automatically. To form |u><u|, supply the conjugate explicitly.


In [ ]:
P = np.outer(u, u.conj())
psi = np.array([1.,0.], dtype=complex)
branch = P @ psi
probability = np.vdot(branch,branch).real
print("P:\n", P, "\nbranch:", branch, "probability:", probability)
assert np.allclose(P @ P, P)
assert np.allclose(P.conj().T, P)
assert np.isclose(probability, 0.5)
conditional = branch / np.sqrt(probability)
assert np.allclose(conditional,u)


## 3.5 Matrices and composition
Verify that order matters. A scales the first component; B rotates a real vector through 90 degrees.


In [ ]:
A = np.diag([2.,1.])
B = np.array([[0.,-1.],[1.,0.]])
v0 = np.array([1.,0.])
print("A then B:", B @ A @ v0)
print("B then A:", A @ B @ v0)
assert np.allclose(B @ A @ v0, [0,2])
assert np.allclose(A @ B @ v0, [0,1])


## 3.6 Check the full unitary condition
Preserving one input does not establish unitarity. Inspect the adjoint product and test several inputs.


In [ ]:
U = np.array([[1,1j],[1j,1]]) / np.sqrt(2)
H = np.array([[1,1],[1,-1]]) / np.sqrt(2)
S = np.diag([1,1j])
def is_unitary(A):
    return A.ndim == 2 and A.shape[0] == A.shape[1] and np.allclose(A.conj().T @ A, np.eye(A.shape[0]))
for name, op in [("U",U),("H",H),("S",S),("scale",A),("projector",P)]:
    print(name, is_unitary(op))
assert is_unitary(U) and is_unitary(H) and is_unitary(S)
assert not is_unitary(A) and not is_unitary(P)
rng = np.random.default_rng(17)
for _ in range(20):
    x = rng.normal(size=2) + 1j*rng.normal(size=2)
    y = rng.normal(size=2) + 1j*rng.normal(size=2)
    assert np.allclose(np.vdot(U@x,U@y), np.vdot(x,y))


## 3.7 Tensor products
Use the order (00,01,10,11): first factor supplies the first label. `np.kron` implements the coordinate tensor product.


In [ ]:
plus = np.array([1,1]) / np.sqrt(2)
phase = np.array([1,1j]) / np.sqrt(2)
joint = np.kron(plus,phase)
print("Joint:", joint)
assert np.allclose(joint, np.array([1,1j,1,1j])/2)
assert np.isclose(np.linalg.norm(joint), 1)
zero, one = np.array([1,0]), np.array([0,1])
result = np.kron(H,np.eye(2)) @ np.kron(zero,one)
assert np.allclose(result, np.array([0,1,0,1])/np.sqrt(2))
assert not np.allclose(np.kron(plus,phase), np.kron(phase,plus))


## Cumulative problems
1. Normalize (1-i,2), calculate its bra and the squared overlap with basis 0.
2. Construct its rank-one projector and check Hermiticity and idempotence.
3. Compare the shapes of its inner, outer and tensor products with itself.
4. For an unnormalized direction w=(2,0), show why outer(w,w*) is not a projector and correct it.
5. Check the two compositions H@S and S@H on basis 0. Do they give identical arrays?
6. Explain which calculations would be needed to interpret a projected vector as a quantum measurement branch.


In [ ]:
w = normalize([1-1j,2])
Pw = np.outer(w,w.conj())
assert np.isclose(abs(np.vdot(zero,w))**2, 1/3)
assert np.allclose(Pw @ Pw,Pw)
assert np.allclose(Pw.conj().T,Pw)
print("Inner shape:", np.asarray(np.vdot(w,w)).shape, "Outer shape:", Pw.shape, "Tensor shape:", np.kron(w,w).shape)
raw = np.array([2.,0.])
fixed = np.outer(raw,raw) / (raw @ raw)
assert np.allclose(fixed @ fixed,fixed)
print("H then S:", S @ H @ zero, "S then H:", H @ S @ zero)
print("All chapter checks passed.")


## Programming references
NumPy documentation: [vdot](https://numpy.org/doc/stable/reference/generated/numpy.vdot.html), [outer](https://numpy.org/doc/stable/reference/generated/numpy.outer.html), [kron](https://numpy.org/doc/stable/reference/generated/numpy.kron.html).
